## 0. Setup & configuration

Every value you might need to edit — file path, sheet names, the reporting
lag — lives here, in one place, at the top. Two reasons: (1) you shouldn't
have to hunt through the notebook to find the one line to change when a
sheet gets renamed, and (2) Jupyter cells don't have to be run top-to-bottom
— nothing stops you from running a later cell before an earlier one, or
re-running one cell after a kernel restart without re-running the rest. If
`SECTOR_SHEET` were defined down in section 3 instead, running section 4
without first running section 3 in the same kernel session would raise
`NameError: name 'SECTOR_SHEET' is not defined`. Keeping config at the top
means it exists the moment the kernel starts, before any loading logic can
possibly ask for it.

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("Dataset.xlsx")

# sheet names confirmed from the "Price"/"ROE"/"EVtoEBITDA"/"MarketCap"/
# "SharesOutstanding" values sheets seen in section 1 — adjust as needed.
FIELD_SHEETS = {
    "price": "Price",
    "roe": "ROE",
    "ev_ebitda": "EVtoEBITDA",
    "market_cap": "MarketCap",
    "shares_outstanding": "SharesOutstanding",
    "debt_equity": "DebtToEquity",  # placeholder name — fix or remove if not present
}
SECTOR_SHEET = "GICS"

REPORTING_LAG_DAYS = 45  # <- tune to your data source's actual reporting lag

MARKET_CAP_FLOOR = 500  # $ millions, matching the MarketCap sheet's units

IC_WINDOW = 12  # months of trailing IC history averaged into each factor's rolling weight

## 1. Inspect the workbook

Bloomberg-style exports often include a live "`- Formulas`" sheet next to a
pasted-values sheet for the same field. The formula sheets show `#N/A` once
opened outside a Bloomberg terminal, so we want the plain-values sheets.
Run this first and confirm the sheet names below match what's actually in
`Dataset.xlsx` before trusting the loader in the next section.

In [12]:
xls = pd.ExcelFile(DATA_PATH)
print(xls.sheet_names)

for name in xls.sheet_names:
    preview = pd.read_excel(DATA_PATH, sheet_name=name, nrows=3)
    print(f"\n=== {name} ===  shape~{preview.shape}")
    print(preview.head(3))

['Price - Formulas', 'Price', 'ROE - Formulas', 'ROE', 'EVtoEBITDA - Formulas', 'EVtoEBITDA', 'MarketCap - Formulas', 'MarketCap', 'GICSClassification - Formulas', 'GICS', 'SharesOutstanding - Formulas', 'SharesOutstanding']

=== Price - Formulas ===  shape~(3, 504)
        DATE  CTAS  RTX  WEC  MAA  AES  FAST  ED  EQIX  LMT  ...  WBD  KVUE  \
0 2026-06-30   NaN  NaN  NaN  NaN  NaN   NaN NaN   NaN  NaN  ...  NaN   NaN   
1 2026-05-31   NaN  NaN  NaN  NaN  NaN   NaN NaN   NaN  NaN  ...  NaN   NaN   
2 2026-04-30   NaN  NaN  NaN  NaN  NaN   NaN NaN   NaN  NaN  ...  NaN   NaN   

   COO  GEV  SOLV  SNDK   Q  CCL  FDXF  DD  
0  NaN  NaN   NaN   NaN NaN  NaN   NaN NaN  
1  NaN  NaN   NaN   NaN NaN  NaN   NaN NaN  
2  NaN  NaN   NaN   NaN NaN  NaN   NaN NaN  

[3 rows x 504 columns]

=== Price ===  shape~(3, 504)
    DATE    CTAS     RTX     WEC     MAA    AES   FAST      ED     EQIX  \
0  46203  170.08  189.73  116.77  138.94  14.66  48.03  110.63  1042.39   
1  46173  171.26  179.66  111.0

## 2. Helpers — consistent tickers & dates

Two gotchas to standardize before anything else:

- **Dates**: the `- Formulas` sheets have their `DATE` column formatted as
  real dates, but the pasted-values sheets sometimes store the *same*
  underlying dates as raw Excel serial numbers (e.g. `46203`) with no date
  number format. If that column isn't coerced explicitly, pandas will treat
  it as an integer and every later date-based join silently breaks.
  `excel_serial_to_date` below handles both cases.
- **Tickers**: uppercase + strip whitespace, so `"ctas "` and `"CTAS"` don't
  become two different tickers after a merge.

In [13]:
def excel_serial_to_date(series: pd.Series) -> pd.Series:
    """Coerce a DATE column to real Timestamps whether it came in as an
    Excel serial number (no date format applied) or was already parsed."""
    if pd.api.types.is_numeric_dtype(series):
        parsed = pd.to_datetime(series, unit="D", origin="1899-12-30")
    else:
        parsed = pd.to_datetime(series)
    # pin to a fixed resolution — pandas can otherwise hand back datetime64[s]
    # vs datetime64[us] depending on the code path taken above, and
    # merge_asof refuses to join keys with mismatched resolutions
    return parsed.astype("datetime64[ns]")


def clean_ticker(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.upper()


def load_field_long(path: Path, sheet_name: str, value_name: str) -> pd.DataFrame:
    """Load a DATE x ticker sheet into tidy long form: date, ticker, <value_name>."""
    df = pd.read_excel(path, sheet_name=sheet_name)
    date_col = df.columns[0]
    df = df.rename(columns={date_col: "date"})
    df["date"] = excel_serial_to_date(df["date"])

    long = df.melt(id_vars="date", var_name="ticker", value_name=value_name)
    long["ticker"] = clean_ticker(long["ticker"])
    long = long.dropna(subset=[value_name])
    return long.sort_values(["ticker", "date"]).reset_index(drop=True)


def load_sector_map(path: Path, sheet_name: str) -> pd.DataFrame:
    """GICS sheet is transposed: rows are attributes (GICS, GICS Sub-Industry),
    columns are tickers. Flip it to one row per ticker."""
    raw = pd.read_excel(path, sheet_name=sheet_name, index_col=0)
    sector = raw.T.reset_index().rename(columns={"index": "ticker"})
    sector.columns = [str(c).strip() for c in sector.columns]
    sector["ticker"] = clean_ticker(sector["ticker"])
    return sector


def ffill_with_reporting_lag(
    price_long: pd.DataFrame,
    fundamental_long: pd.DataFrame,
    value_name: str,
    lag_days: int,
) -> pd.DataFrame:
    """Attach a fundamental to each price date using as-of (backward)
    forward-fill, but only once it would actually have been public:
    available_date = report/period date + lag_days."""
    fundamental_long = fundamental_long.copy()
    fundamental_long["available_date"] = (
        fundamental_long["date"] + pd.Timedelta(days=lag_days)
    ).astype("datetime64[ns]")

    merged_parts = []
    for ticker, price_grp in price_long.groupby("ticker"):
        fund_grp = (
            fundamental_long.loc[fundamental_long["ticker"] == ticker, ["available_date", value_name]]
            .sort_values("available_date")
        )
        price_grp = price_grp.sort_values("date")
        merged = pd.merge_asof(
            price_grp, fund_grp, left_on="date", right_on="available_date", direction="backward"
        )
        merged_parts.append(merged.drop(columns="available_date"))

    return pd.concat(merged_parts, ignore_index=True)

## 3. Load each field's sheet

`FIELD_SHEETS` and `SECTOR_SHEET` are set in section 0 — edit them there if
a name doesn't match what you saw printed in section 1 (in particular,
confirm the debt/equity sheet name, or delete that line if you're not using
it). Anything ending in `- Formulas` is skipped on purpose.

In [14]:
fields = {}
for field, sheet in FIELD_SHEETS.items():
    if sheet not in xls.sheet_names:
        print(f"skipping '{field}': sheet '{sheet}' not found in workbook")
        continue
    fields[field] = load_field_long(DATA_PATH, sheet, field)
    print(f"{field:<20} <- '{sheet}'  rows={len(fields[field]):>6}  "
          f"tickers={fields[field]['ticker'].nunique()}  "
          f"dates={fields[field]['date'].min().date()} .. {fields[field]['date'].max().date()}")

# ticker consistency check across sheets
ticker_sets = {f: set(df["ticker"]) for f, df in fields.items()}
common_tickers = set.intersection(*ticker_sets.values())
for f, s in ticker_sets.items():
    extra = s - common_tickers
    if extra:
        print(f"WARNING: '{f}' has tickers not shared by every sheet: {sorted(extra)}")
print("common tickers:", sorted(common_tickers))

price                <- 'Price'  rows= 59035  tickers=503  dates=2016-06-30 .. 2026-06-30
roe                  <- 'ROE'  rows= 56316  tickers=495  dates=2016-06-30 .. 2026-06-30
ev_ebitda            <- 'EVtoEBITDA'  rows= 52819  tickers=460  dates=2016-06-30 .. 2026-06-30
market_cap           <- 'MarketCap'  rows= 59071  tickers=503  dates=2016-06-30 .. 2026-06-30
shares_outstanding   <- 'SharesOutstanding'  rows= 59773  tickers=503  dates=2016-06-30 .. 2026-06-30
skipping 'debt_equity': sheet 'DebtToEquity' not found in workbook
common tickers: ['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AJG', 'AKAM', 'ALB', 'ALGN', 'ALLE', 'AMAT', 'AMCR', 'AMD', 'AME', 'AMGN', 'AMP', 'AMT', 'AMZN', 'ANET', 'AON', 'AOS', 'APA', 'APD', 'APH', 'APO', 'APP', 'APTV', 'ARE', 'ARES', 'ATO', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXON', 'AXP', 'BA', 'BALL', 'BAX', 'BBY', 'BDX', 'BEN', 'BF.B', 'BG', 'BIIB', 'BKNG', 'BKR', 'BLDR', 'BLK', 'BMY', 'BR', 'BRK.B', '

In [15]:
sector_map = load_sector_map(DATA_PATH, SECTOR_SHEET)
print(sector_map)

    ticker                    GICS  \
0     CTAS             Industrials   
1      RTX             Industrials   
2      WEC               Utilities   
3      MAA             Real Estate   
4      AES               Utilities   
..     ...                     ...   
498   SNDK  Information Technology   
499      Q  Information Technology   
500    CCL  Consumer Discretionary   
501   FDXF             Industrials   
502     DD               Materials   

                                GICS Sub-Industry  
0                    Diversified Support Services  
1                             Aerospace & Defense  
2                              Electric Utilities  
3                  Multi-Family Residential REITs  
4    Independent Power Producers & Energy Traders  
..                                            ...  
498    Technology Hardware, Storage & Peripherals  
499           Semiconductor Materials & Equipment  
500                Hotels, Resorts & Cruise Lines  
501                   C

## 4. Align fundamentals onto price dates

Price is the backbone — it's the frequency the model actually trades on.
Each fundamental is forward-filled *as of* the price date, but only after
adding `REPORTING_LAG_DAYS` (set in section 0): a quarter that ended on
`date` wasn't public knowledge until roughly `date + lag`, so shifting
`available_date` forward before the as-of join keeps the panel from leaking
future information into past price rows (look-ahead bias). Adjust the lag
in section 0 to match how your source reports fundamentals (45 days is a
common rough estimate for U.S. 10-Q filings; use whatever your actual
filing-lag convention is).

In [16]:
panel = fields["price"][fields["price"]["ticker"].isin(common_tickers)].copy()

fundamental_fields = [f for f in fields if f != "price"]
for f in fundamental_fields:
    fund = fields[f][fields[f]["ticker"].isin(common_tickers)]
    panel = ffill_with_reporting_lag(panel, fund, f, REPORTING_LAG_DAYS)

panel = panel.merge(sector_map, on="ticker", how="left")
panel = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
panel.head(10)

,date,ticker,price,roe,ev_ebitda,market_cap,shares_outstanding,GICS,GICS Sub-Industry
0,2016-06-30,A,44.36,NaN,NaN,NaN,NaN,Health Care,Life Sciences Tools & Services
1,2016-07-31,A,48.11,NaN,NaN,NaN,NaN,Health Care,Life Sciences Tools & Services
2,2016-08-31,A,46.98,11.033654,15.040648,14440.156259,331.459,Health Care,Life Sciences Tools & Services
3,2016-09-30,A,47.09,11.318968,15.040648,15660.863771,331.459,Health Care,Life Sciences Tools & Services
4,2016-10-31,A,43.57,11.318968,15.040648,15293.023560,331.459,Health Care,Life Sciences Tools & Services
5,2016-11-30,A,43.98,11.318968,15.040648,15275.290159,331.459,Health Care,Life Sciences Tools & Services
6,2016-12-31,A,45.56,10.986920,14.216426,14133.454776,324.000,Health Care,Life Sciences Tools & Services
7,2017-01-31,A,48.97,10.986920,14.216426,14266.452581,324.000,Health Care,Life Sciences Tools & Services
8,2017-02-28,A,51.30,10.986920,14.216426,14658.838677,324.000,Health Care,Life Sciences Tools & Services
9,2017-03-31,A,52.87,12.155358,14.216426,15755.999259,324.000,Health Care,Life Sciences Tools & Services


## 5. Sanity checks

Missing values right after the panel's start date are expected (there's no
prior fundamental to forward-fill from yet). Missing values in the middle
or end of the series are not — that's a sign a ticker or date mismatch
slipped through.

In [17]:
print("panel shape:", panel.shape)
print("\nmissing values per column:\n", panel.isna().sum())

print("\nrows per ticker:\n", panel.groupby("ticker").size())

# show where each fundamental first becomes non-null per ticker, to eyeball
# that forward-fill + lag look reasonable against the raw source dates
for f in fundamental_fields:
    first_valid = panel.dropna(subset=[f]).groupby("ticker")["date"].min()
    print(f"\nfirst non-null '{f}' by ticker:\n{first_valid}")

panel shape: (53224, 9)

missing values per column:
 date                     0
ticker                   0
price                    0
roe                   1035
ev_ebitda             1649
market_cap             904
shares_outstanding     865
GICS                     0
GICS Sub-Industry        0
dtype: int64

rows per ticker:
 ticker
A       121
AAPL    121
ABBV    121
ABNB     67
ABT     121
       ... 
XYZ     121
YUM     121
ZBH     121
ZBRA    121
ZTS     121
Length: 453, dtype: int64

first non-null 'roe' by ticker:
ticker
A      2016-08-31
AAPL   2016-08-31
ABBV   2016-08-31
ABNB   2021-02-28
ABT    2016-08-31
          ...    
XYZ    2016-08-31
YUM    2016-08-31
ZBH    2016-08-31
ZBRA   2016-08-31
ZTS    2016-08-31
Name: date, Length: 453, dtype: datetime64[ns]

first non-null 'ev_ebitda' by ticker:
ticker
A      2016-08-31
AAPL   2016-08-31
ABBV   2016-08-31
ABNB   2022-02-28
ABT    2016-08-31
          ...    
XYZ    2018-02-28
YUM    2016-08-31
ZBH    2016-08-31
ZBRA   2016-08

## Factor Implementation

### Value, Quality, Low Volatility

All four factors are defined so that **higher = better** — this matters
once these get z-scored and combined into a composite score later, since a
mismatched sign would have that factor pulling the composite the wrong way.

- **Value (EV/EBITDA)**: a *lower* EV/EBITDA means a cheaper stock, so the
  raw field gets negated. Negation (rather than `1/x`) is deliberate — a
  reciprocal blows up for names with EV/EBITDA near zero, which negation
  doesn't.
- **Quality (ROE)**: higher ROE is already "better," so no inversion.
- **Low volatility**: computed from monthly *returns*, not price directly —
  `price.pct_change()` per ticker, then a trailing 12-month rolling std dev
  of those returns, then negated (lower realized vol → higher factor
  value). Same as momentum, both the return calculation and the rolling
  window are grouped by ticker so one company's prices/returns never bleed
  into another's.

In [18]:
def momentum_12_1(panel: pd.DataFrame, price_col: str = "price", ticker_col: str = "ticker") -> pd.Series:
    """12-1 month momentum: (price[t-1] / price[t-13]) - 1.

    Shifts are taken within each ticker's own price history (via groupby)
    so one ticker's prices never leak into another's calculation at the
    boundary between tickers. Assumes `panel` is sorted by [ticker, date]
    with no missing months per ticker — otherwise shift(13) grabs "13 rows
    back" rather than "13 calendar months back."
    """
    price_by_ticker = panel.groupby(ticker_col)[price_col]
    price_t_minus_1 = price_by_ticker.shift(1)
    price_t_minus_13 = price_by_ticker.shift(13)
    return price_t_minus_1 / price_t_minus_13 - 1

def value_factor(panel: pd.DataFrame, ev_ebitda_col: str = "ev_ebitda") -> pd.Series:
    """Value factor: -1 * EV/EBITDA. A lower EV/EBITDA means a cheaper
    stock, so it's negated here to match the "higher = better" convention
    used for every factor."""
    return -1 * panel[ev_ebitda_col]


def quality_factor(panel: pd.DataFrame, roe_col: str = "roe") -> pd.Series:
    """Quality factor: ROE as-is. Higher ROE already means better quality,
    so no inversion — kept as its own function so leverage or other quality
    signals can be folded in later without touching call sites."""
    return panel[roe_col]


def low_vol_factor(
    panel: pd.DataFrame,
    price_col: str = "price",
    ticker_col: str = "ticker",
    window: int = 12,
) -> pd.Series:
    """Low-volatility factor: -1 * trailing `window`-month std dev of
    monthly returns. Returns and the rolling std dev are both computed
    within each ticker's own history (via groupby) so one ticker's
    prices/returns never leak into another's calculation. Negated so a
    higher factor value means lower realized volatility ("better").
    Assumes `panel` is sorted by [ticker, date] with no missing months per
    ticker, same as momentum_12_1.
    """
    returns = panel.groupby(ticker_col)[price_col].pct_change()
    trailing_vol = (
        returns.groupby(panel[ticker_col])
        .rolling(window)
        .std()
        .reset_index(level=0, drop=True)
    )
    return -1 * trailing_vol

In [19]:
panel["mom_12_1"] = momentum_12_1(panel)
panel["value"] = value_factor(panel)
panel["quality"] = quality_factor(panel)
panel["low_vol"] = low_vol_factor(panel)

# spot-check one ticker by hand: mom_12_1 at row t should equal
# price at t-1 divided by price at t-13, minus 1
check = panel[panel["ticker"] == "AAPL"].reset_index(drop=True)
check["manual_check"] = check["price"].shift(1) / check["price"].shift(13) - 1
print("\nmatches:", check["mom_12_1"].equals(check["manual_check"]))

# spot-check low_vol by hand for one ticker: manually compute monthly
# returns and a trailing 12-month std dev, and confirm it matches
check = panel[panel["ticker"] == "AAPL"].reset_index(drop=True)
manual_returns = check["price"].pct_change()
check["manual_low_vol"] = -1 * manual_returns.rolling(12).std()
print("\nlow_vol matches manual calc:", check["low_vol"].equals(check["manual_low_vol"]))




matches: True

low_vol matches manual calc: True


## 6. Market-cap floor filter

Applied **per date**, not once at the start — a stock can cross $500M in
either direction over a 10-year window, so this has to be a per-row
eligibility flag rather than a one-time drop of tickers that happened to be
small at t=0 (which would also incorrectly keep a stock that later shrank
below the floor). Rows where `market_cap` is still `NaN` (before a ticker's
first reported fundamental) come out as `False` too — not because they're
known to be small-cap, but because eligibility can't be confirmed yet. Both
cases mean "exclude from that date's cross-section," which is the only
thing this flag needs to express for now.

In [20]:
panel["above_cap_floor"] = panel["market_cap"] >= MARKET_CAP_FLOOR

missing_cap = panel["market_cap"].isna().sum()
below_floor = (panel["market_cap"] < MARKET_CAP_FLOOR).sum()
print(f"excluded rows — missing market cap: {missing_cap}, below ${MARKET_CAP_FLOOR}M floor: {below_floor}")

# eligible universe size over time — should track close to the full
# universe, dipping only for genuinely small names
coverage = pd.DataFrame({
    "eligible": panel.groupby("date")["above_cap_floor"].sum(),
    "total": panel.groupby("date")["ticker"].nunique(),
})
coverage["pct_eligible"] = (coverage["eligible"] / coverage["total"]).round(3)
print(coverage.tail(10))

print("\ntickers ever excluded by the cap floor (missing data or sub-floor):")
print(sorted(panel.loc[~panel["above_cap_floor"], "ticker"].unique()))

excluded rows — missing market cap: 904, below $500M floor: 0
            eligible  total  pct_eligible
date                                     
2025-09-30       452    452         1.000
2025-10-31       452    453         0.998
2025-11-30       452    453         0.998
2025-12-31       453    453         1.000
2026-01-31       453    453         1.000
2026-02-28       453    453         1.000
2026-03-31       453    453         1.000
2026-04-30       453    453         1.000
2026-05-31       453    453         1.000
2026-06-30       453    453         1.000

tickers ever excluded by the cap floor (missing data or sub-floor):
['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADSK', 'AEE', 'AEP', 'AES', 'AJG', 'AKAM', 'ALB', 'ALGN', 'ALLE', 'AMAT', 'AMD', 'AME', 'AMGN', 'AMP', 'AMT', 'AMZN', 'ANET', 'AON', 'AOS', 'APA', 'APD', 'APH', 'APO', 'APP', 'APTV', 'ARE', 'ARES', 'ATO', 'AVB', 'AVGO', 'AVY', 'AWK', 'AXON', 'AXP', 'BA', 'BALL', 'BAX', 'BBY', 'BDX', 'BEN', 

## 7. Cross-sectional z-scoring, within sector, per date

A raw factor value is meaningless on its own — an EV/EBITDA of 15 doesn't
tell you if a stock is cheap or expensive without something to compare it
to. Z-scoring converts each raw value into "how many standard deviations
above/below average, *within its own comparison group*, on this date."

Two design choices worth calling out:

- **Within sector, not across the whole universe.** Utilities and tech
  stocks trade at structurally different EV/EBITDA multiples and carry
  different volatility, for reasons that have nothing to do with which one
  is the better investment. Z-scoring within `GICS` sector means a utility
  is only compared against other utilities, so the score reflects "cheap
  *for a utility*" rather than "cheap relative to Nvidia" — sector-neutral,
  matching the reference doc.
- **Only cap-floor-eligible stocks set the mean/std.** A sub-floor or
  data-missing stock (`above_cap_floor == False`) shouldn't pull the
  sector's average around, since it wouldn't be investable anyway — it's
  excluded from the population the z-score is computed against, not just
  excluded from the final portfolio.

Both a missing factor value and an ineligible stock come back as `NaN`
here, never `0` — a `0` z-score means "exactly average," which is a false
signal for a stock that has no data at all. That's what "exclude, don't
zero-fill" (reference step 7) means in practice.

In [21]:
FACTOR_COLUMNS = ["mom_12_1", "value", "quality", "low_vol"]


def zscore_within_sector(
    panel: pd.DataFrame,
    factor_col: str,
    date_col: str = "date",
    sector_col: str = "GICS",
    eligible_col: str = "above_cap_floor",
) -> pd.Series:
    """Cross-sectional z-score of `factor_col`, computed within each
    (date, sector) group, using only cap-floor-eligible rows to set the
    group's mean/std. Ineligible rows and rows with a missing factor value
    both come back NaN — excluded from the score, not zero-filled.
    """
    # NaN out ineligible rows *before* grouping, so they don't influence
    # the mean/std either — pandas' mean()/std() skip NaN by default
    eligible_values = panel[factor_col].where(panel[eligible_col])

    group_keys = [panel[date_col], panel[sector_col]]
    group_mean = eligible_values.groupby(group_keys).transform("mean")
    group_std = eligible_values.groupby(group_keys).transform("std")

    return (eligible_values - group_mean) / group_std


for col in FACTOR_COLUMNS:
    panel[f"z_{col}"] = zscore_within_sector(panel, col)

z_cols = [f"z_{c}" for c in FACTOR_COLUMNS]
print(panel[["date", "ticker", "GICS", "above_cap_floor"] + FACTOR_COLUMNS + z_cols].tail(10))

            date ticker         GICS  above_cap_floor  mom_12_1      value  \
53214 2025-09-30    ZTS  Health Care             True -0.147637 -20.236406   
53215 2025-10-31    ZTS  Health Care             True -0.251100 -20.236406   
53216 2025-11-30    ZTS  Health Care             True -0.194037 -20.236406   
53217 2025-12-31    ZTS  Health Care             True -0.268588 -20.236406   
53218 2026-01-31    ZTS  Health Care             True -0.227767 -20.236406   
53219 2026-02-28    ZTS  Health Care             True -0.269631 -15.323291   
53220 2026-03-31    ZTS  Health Care             True -0.216097 -15.323291   
53221 2026-04-30    ZTS  Health Care             True -0.282053 -15.323291   
53222 2026-05-31    ZTS  Health Care             True -0.264898 -15.323291   
53223 2026-06-30    ZTS  Health Care             True -0.539287 -15.323291   

         quality   low_vol  z_mom_12_1   z_value  z_quality  z_low_vol  
53214  52.539474 -0.058526   -0.419776  0.002615  -0.072665   0.7873

In [22]:
# sanity check: for one (date, sector) group, the eligible members' z-scores
# should have mean ~0 and std ~1 by construction
sample_date = panel["date"].max()
sample_sector = panel.loc[panel["date"] == sample_date, "GICS"].mode().iloc[0]
sample = panel[
    (panel["date"] == sample_date)
    & (panel["GICS"] == sample_sector)
    & panel["above_cap_floor"]
]
print(f"{sample_sector} on {sample_date.date()}, n={len(sample)}")
print(sample[["ticker", "mom_12_1", "z_mom_12_1"]])
print("\nz_mom_12_1 mean (should be ~0):", sample["z_mom_12_1"].mean())
print("z_mom_12_1 std (should be ~1):", sample["z_mom_12_1"].std())

Industrials on 2026-06-30, n=79
      ticker  mom_12_1  z_mom_12_1
1155     ADP -0.318527   -1.057549
2244    ALLE -0.088507   -0.626764
2692     AME  0.263567    0.032607
3539     AOS -0.118022   -0.682040
5175    AXON -0.401994   -1.213867
...      ...       ...         ...
49742   VRSK -0.442955   -1.290581
49838    VRT  1.925137    3.144424
50561    WAB  0.290826    0.083659
51408     WM -0.122463   -0.690358
52618    XYL -0.130911   -0.706178

[79 rows x 3 columns]

z_mom_12_1 mean (should be ~0): -1.1242764806330699e-17
z_mom_12_1 std (should be ~1): 0.9999999999999998


## 8. Composite score (equal-weight)

Average the four z-scores per stock per date. `DataFrame.mean(axis=1)`
skips `NaN`s by default (`skipna=True`), so a stock missing one factor
still gets a composite from the other three instead of being penalized
with an implicit zero for the missing one — consistent with "exclude, not
zero-fill." `n_factors_available` is tracked alongside so you can see —
and later decide whether to filter on — how much of each composite score
actually rests on real data versus a single available factor.

In [23]:
panel["composite_score"] = panel[z_cols].mean(axis=1, skipna=True)
panel["n_factors_available"] = panel[z_cols].notna().sum(axis=1)

print(panel[["date", "ticker"] + z_cols + ["composite_score", "n_factors_available"]].tail(10))

print("\ndistribution of n_factors_available:")
print(panel["n_factors_available"].value_counts().sort_index())

print("\nrows with a composite score but 0 factors available (should be none):",
      (panel["composite_score"].notna() & (panel["n_factors_available"] == 0)).sum())

            date ticker  z_mom_12_1   z_value  z_quality  z_low_vol  \
53214 2025-09-30    ZTS   -0.419776  0.002615  -0.072665   0.787366   
53215 2025-10-31    ZTS   -0.796223  0.002615  -0.072665   1.029993   
53216 2025-11-30    ZTS   -0.861878  0.000454  -0.082274   0.899046   
53217 2025-12-31    ZTS   -1.333378 -0.005542  -0.082294   0.946473   
53218 2026-01-31    ZTS   -1.350287 -0.005542  -0.082294   0.896932   
53219 2026-02-28    ZTS   -1.332751  0.254450  -0.142952   0.818859   
53220 2026-03-31    ZTS   -1.300893  0.237027  -0.142946   0.886595   
53221 2026-04-30    ZTS   -1.326595  0.237027  -0.142946   0.852281   
53222 2026-05-31    ZTS   -1.247089  0.230596  -0.141943  -0.631326   
53223 2026-06-30    ZTS   -1.982691  0.227045  -0.141941  -0.409632   

       composite_score  n_factors_available  
53214         0.074385                    4  
53215         0.040930                    4  
53216        -0.011163                    4  
53217        -0.118685            

## 9. IC-weighted composite (rolling)

Equal-weight treats all four factors as equally useful, which is a
reasonable starting assumption but not something we actually know. IC
(Information Coefficient) weighting instead measures, historically, how
well each factor's z-score actually lined up with what happened next, and
leans more on the factors that did.

This needs three new pieces, in order:

1. **A forward return** — the thing each factor is trying to predict.
   `price[t+1] / price[t] - 1`, per ticker.
2. **A per-date IC** — the cross-sectional (Spearman rank) correlation
   between a factor's z-score at date `t` and the forward return at `t`,
   across every stock that has both. One number per factor per date: "how
   predictive was this factor, for the move that just happened."
3. **A rolling, *lagged* average of IC** — this is the step that actually
   prevents lookahead. `IC_t` requires `price[t+1]`, which isn't known yet
   at the moment a portfolio is being formed at date `t`. So the weight
   used *at* `t` has to come from a rolling average of `IC_{t-1}` back
   through `IC_{t-window}` — never `IC_t` itself. Skipping the lag would
   mean using next month's outcome to decide this month's weights, which
   is exactly the kind of leak the reporting lag in section 4 was built to
   avoid, just showing up in a different part of the pipeline.

Negative rolling IC gets clipped to zero rather than flipped to a negative
weight — a factor with no recent predictive power gets excluded from that
date's blend, not inverted. Inverting a factor based on a noisy trailing
correlation estimate is a good way to overfit to recent noise.

In [24]:
def compute_forward_return(panel: pd.DataFrame, price_col: str = "price", ticker_col: str = "ticker") -> pd.Series:
    """Forward 1-month return: price[t+1] / price[t] - 1, per ticker.

    This is a label, not a feature — it's only ever used to score how
    predictive a factor *was* in section 9's IC calculation, never used
    directly in a stock's own composite score. shift(-1) looks *ahead* one
    row within each ticker's group, which is exactly why this column must
    never be joined back into a stock's own factor inputs.
    """
    next_price = panel.groupby(ticker_col)[price_col].shift(-1)
    return next_price / panel[price_col] - 1


panel["fwd_return"] = compute_forward_return(panel)

# every ticker's *last* row should be NaN — there's no "next month" price
# to look ahead to yet, which is the mirror image of momentum's warm-up NaNs
print(panel.groupby("ticker").tail(1)["fwd_return"].isna().all())

True


In [25]:
def compute_ic_series(panel: pd.DataFrame, z_col: str, forward_return_col: str = "fwd_return", date_col: str = "date") -> pd.Series:
    """Cross-sectional rank IC per date: Spearman correlation between a
    factor's z-score and the forward return it's trying to predict, across
    every stock that has both non-null on that date. Rank correlation
    (Spearman), not linear (Pearson) — standard in factor investing, and
    far less sensitive to a handful of extreme monthly return outliers
    than a linear correlation would be.

    Looped explicitly over each date rather than groupby().apply(...) —
    slower, but the intermediate result for any one date is easy to
    reproduce by hand if a number ever looks wrong, same tradeoff as the
    per-ticker loop in ffill_with_reporting_lag.
    """
    ic_by_date = {}
    for date, group in panel.groupby(date_col):
        ic_by_date[date] = group[z_col].corr(group[forward_return_col], method="spearman")
    return pd.Series(ic_by_date).sort_index()


ic_df = pd.DataFrame({col: compute_ic_series(panel, col) for col in z_cols})
print(ic_df.describe())

# rolling mean IC, then shift(1) so the value attached to date t only ever
# reflects IC measured at t-1 and earlier — never t itself
rolling_ic_df = ic_df.rolling(IC_WINDOW).mean().shift(1)
print("\nrolling (lagged) IC, most recent dates:")
print(rolling_ic_df.tail())

       z_mom_12_1     z_value   z_quality   z_low_vol
count  107.000000  118.000000  118.000000  108.000000
mean     0.011213    0.001428    0.004965   -0.020867
std      0.152768    0.127359    0.073152    0.149459
min     -0.419110   -0.297449   -0.199020   -0.336425
25%     -0.107100   -0.076213   -0.047260   -0.128891
50%      0.028047    0.000895    0.007915   -0.007043
75%      0.118989    0.084586    0.049882    0.083798
max      0.330621    0.308996    0.180919    0.301437

rolling (lagged) IC, most recent dates:
            z_mom_12_1   z_value  z_quality  z_low_vol
2026-02-28   -0.014628  0.047930  -0.014523  -0.076447
2026-03-31    0.013692  0.044570  -0.016616  -0.095198
2026-04-30    0.024419  0.052705  -0.015144  -0.096784
2026-05-31    0.031300  0.063867  -0.019966  -0.094175
2026-06-30    0.046220  0.058134  -0.006572  -0.070651


### Turning rolling IC into weights, and combining

`compute_ic_weights` clips negative IC to zero and renormalizes the
positive values to sum to 1 across the four factors, per date. Two
situations both fall back to flat 25%-each weighting: the first ~`IC_WINDOW`
months of history (rolling IC is still `NaN` — not enough trailing data
yet), and any date where every factor's rolling IC happens to be zero or
negative. Both are "we don't have a reliable signal to overweight anything
with," and equal-weight is the reasonable default for that case — which
means the IC-weighted composite is *identical* to the equal-weight one for
that initial warm-up stretch, by construction, not by coincidence.

Combining the weights with each stock's z-scores has one gotcha worth
flagging explicitly: **zeroing out a weight doesn't stop a missing
z-score from poisoning the sum**, because `NaN * 0` is still `NaN`, not
`0` (that's IEEE-754 floating point, not a pandas quirk). So the missing
factor value itself has to be replaced with `0` too, *in addition to*
zeroing its weight — the zeroed weight is what keeps that `0` from
actually contributing anything once divided back out by the (correctly
shrunk) sum of weights.

In [26]:
def compute_ic_weights(rolling_ic_by_factor: pd.DataFrame) -> pd.DataFrame:
    """Turn each factor's rolling (lagged) IC into a per-date weight.
    Negative IC is clipped to zero, and the positive values renormalized
    to sum to 1 — see markdown above for why, and for the equal-weight
    fallback used when weight_sum is 0 (warm-up, or no factor currently
    has positive predictive power)."""
    n_factors = rolling_ic_by_factor.shape[1]
    clipped = rolling_ic_by_factor.clip(lower=0)
    weight_sum = clipped.sum(axis=1)

    weights = clipped.div(weight_sum, axis=0)
    weights = weights.where(weight_sum > 0, 1 / n_factors)
    return weights


weights_df = compute_ic_weights(rolling_ic_df)
print(weights_df.tail())
print("\nweights sum to 1 each date:", np.allclose(weights_df.sum(axis=1), 1.0))


def combine_weighted(panel: pd.DataFrame, z_cols: list[str], weight_cols: list[str]) -> pd.Series:
    """Weighted average of each stock's z-scores, using per-date,
    per-factor weights already broadcast onto `panel` (one weight column
    per z-score column, same order). Missing z-scores are excluded and the
    remaining weights renormalized, same "exclude, don't zero-fill"
    principle as the equal-weight composite — see markdown above for the
    NaN * 0 = NaN subtlety this specifically works around.

    Uses .to_numpy() rather than DataFrame arithmetic: pandas aligns two
    DataFrames' columns by *label* during arithmetic, but z_cols and
    weight_cols intentionally have different names — dropping to numpy
    sidesteps that entirely and just multiplies position-for-position,
    which is correct here only because both lists were built in the same
    order.
    """
    z = panel[z_cols].to_numpy()
    w = panel[weight_cols].to_numpy()

    is_missing = np.isnan(z)
    w_masked = np.where(is_missing, 0.0, w)
    z_filled = np.where(is_missing, 0.0, z)

    weight_sum = w_masked.sum(axis=1)
    contribution = (z_filled * w_masked).sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        result = contribution / weight_sum
    result = np.where(weight_sum > 0, result, np.nan)
    return pd.Series(result, index=panel.index)


weight_cols = []
for col in z_cols:
    wcol = f"w_{col}"
    panel[wcol] = panel["date"].map(weights_df[col])
    weight_cols.append(wcol)

panel["composite_ic_weighted"] = combine_weighted(panel, z_cols, weight_cols)
print(panel[["date", "ticker", "composite_score", "composite_ic_weighted"]].tail(10))

            z_mom_12_1   z_value  z_quality  z_low_vol
2026-02-28    0.000000  1.000000        0.0        0.0
2026-03-31    0.235006  0.764994        0.0        0.0
2026-04-30    0.316618  0.683382        0.0        0.0
2026-05-31    0.328895  0.671105        0.0        0.0
2026-06-30    0.442914  0.557086        0.0        0.0

weights sum to 1 each date: True
            date ticker  composite_score  composite_ic_weighted
53214 2025-09-30    ZTS         0.074385              -0.237017
53215 2025-10-31    ZTS         0.040930               0.002615
53216 2025-11-30    ZTS        -0.011163               0.000454
53217 2025-12-31    ZTS        -0.118685              -0.005542
53218 2026-01-31    ZTS        -0.135298              -0.005542
53219 2026-02-28    ZTS        -0.100599               0.254450
53220 2026-03-31    ZTS        -0.080055              -0.124394
53221 2026-04-30    ZTS        -0.095058              -0.258044
53222 2026-05-31    ZTS        -0.447441              -0.255

In [27]:
# --- sanity checks ---

# during the warm-up window (before IC_WINDOW months of IC history exist),
# the two composites should be exactly equal, by construction
warmup_dates = ic_df.index[:IC_WINDOW]
warmup_rows = panel[panel["date"].isin(warmup_dates)]
print("composites match during warm-up:",
      np.allclose(warmup_rows["composite_score"], warmup_rows["composite_ic_weighted"], equal_nan=True))

# after warm-up, they should generally differ (IC-weighting is doing something)
# but stay reasonably correlated (both are still blends of the same 4 factors)
post_warmup = panel[~panel["date"].isin(warmup_dates)].dropna(subset=["composite_score", "composite_ic_weighted"])
print("correlation between the two composites post-warmup:",
      post_warmup["composite_score"].corr(post_warmup["composite_ic_weighted"]))

# factor efficacy summary — mean raw (unlagged) IC and % of dates with a
# positive IC per factor, i.e. "how predictive was each factor overall"
print("\nfactor efficacy (mean IC, % positive months):")
print(pd.DataFrame({
    "mean_ic": ic_df.mean(),
    "pct_positive": (ic_df > 0).mean(),
}))

composites match during warm-up: True
correlation between the two composites post-warmup: 0.6106485769135318

factor efficacy (mean IC, % positive months):
             mean_ic  pct_positive
z_mom_12_1  0.011213      0.520661
z_value     0.001428      0.487603
z_quality   0.004965      0.528926
z_low_vol  -0.020867      0.396694
